# REM sleep without atonia: CAP extraction and scoring

Runs entirely in the cloud. Each recording is streamed from PhysioNet's open
mirror into this machine's temporary disk, reduced to a small EMG-envelope file
that is saved to **Google Drive** (`MyDrive/rbd-rwa`), and the raw file is
deleted. Progress is recorded after every recording, so if the session
disconnects just run all cells again and it continues where it stopped.

Run order: **Runtime -> Run all**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/rbd-rwa/src
%cd /content/rbd-rwa

In [ ]:
%%writefile /content/rbd-rwa/src/__init__.py

In [ ]:
%%writefile /content/rbd-rwa/src/config.py
"""Paths that work identically on Colab, locally, or anywhere else.

The guiding rule: RAW recordings are disposable, DERIVED data is precious.
Raw PSG is hundreds of MB per night and lives in scratch (Colab's ephemeral
disk); the extracted envelope + hypnogram is a few MB and lives in a durable
directory (Google Drive), so a disconnected session never loses real work.
"""
import os
from pathlib import Path


def _first_existing(*candidates):
    for c in candidates:
        if c and Path(c).exists():
            return Path(c)
    return None


ON_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

# Durable storage: Drive on Colab, the project folder otherwise.
DRIVE = _first_existing(
    os.environ.get("RBD_DRIVE_DIR"),
    "/content/drive/MyDrive",
)
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DURABLE = Path(os.environ.get("RBD_DURABLE_DIR",
                              (DRIVE / "rbd-rwa") if DRIVE else PROJECT_ROOT / "data"))

# Scratch: fast, large, and wiped when the session ends. Raw EDFs only.
SCRATCH = Path(os.environ.get("RBD_SCRATCH_DIR",
                              "/content/scratch" if ON_COLAB else PROJECT_ROOT / "data" / "raw"))

DERIVED   = DURABLE / "derived"      # one compact .npz per recording
MANIFESTS = DURABLE / "manifests"    # progress records, so a restart resumes
FIGURES   = DURABLE / "figures"
RESULTS   = DURABLE / "results"

for d in (DERIVED, MANIFESTS, FIGURES, RESULTS, SCRATCH):
    d.mkdir(parents=True, exist_ok=True)

# Envelope sampling rate. Phasic RWA bursts are >= 0.1 s, so 50 Hz (20 ms bins)
# gives 5 samples across the shortest event a method needs to see, while being
# ~100x smaller than the raw signal.
ENV_FS = 50

# EMG band. 10 Hz high-pass removes movement/drift and ECG baseline; 100 Hz is
# the conventional upper edge, lowered automatically when Nyquist forbids it.
EMG_BAND = (10.0, 100.0)


def describe():
    print(f"on Colab : {ON_COLAB}")
    print(f"durable  : {DURABLE}   (derived data, survives disconnects)")
    print(f"scratch  : {SCRATCH}   (raw EDFs, disposable)")

In [ ]:
%%writefile /content/rbd-rwa/src/edf.py
"""Minimal EDF/EDF+ reader that returns each channel at its NATIVE rate.

Why not MNE: `mne.io.read_raw_edf` resamples every channel to the highest rate
in the file. CAP stores chin EMG at 256-512 Hz but leg EMG at 128 Hz, so the
leg channels silently came back upsampled to 512 Hz. Reading the samples
directly keeps each channel's true rate, reads only the channels asked for,
and needs nothing beyond numpy - which matters on hosted notebooks where
installing packages is not always possible.

Values are returned in microvolts (from the header's physical dimension), so
amplitudes are comparable across files that store uV, mV or V.
"""
from dataclasses import dataclass
from pathlib import Path

import numpy as np

_TO_UV = {"uv": 1.0, "µv": 1.0, "mv": 1e3, "v": 1e6, "nv": 1e-3}


@dataclass
class Channel:
    index: int
    label: str
    dim: str
    fs: float
    n_per_record: int
    gain: float          # physical units per digital unit
    offset: float        # physical value at digital zero
    prefilter: str

    @property
    def uv_scale(self):
        """Factor from physical units to microvolts, or None if not a voltage."""
        return _TO_UV.get(self.dim.strip().lower().replace("μ", "µ"))


@dataclass
class EdfHeader:
    path: Path
    n_records: int
    record_s: float
    header_bytes: int
    start_date: str
    start_time: str
    channels: list

    @property
    def duration_s(self):
        return self.n_records * self.record_s

    def find(self, label):
        for c in self.channels:
            if c.label == label:
                return c
        raise KeyError(label)


def read_header(path) -> EdfHeader:
    path = Path(path)
    with open(path, "rb") as f:
        h = f.read(256)
        ns = int(h[252:256])
        s = f.read(256 * ns)

    def fields(width, start):
        return [s[start + i * width: start + (i + 1) * width]
                .decode("latin-1").strip() for i in range(ns)], start + width * ns

    labels, o = fields(16, 0)
    _, o = fields(80, o)                          # transducer
    dims, o = fields(8, o)
    pmin, o = fields(8, o)
    pmax, o = fields(8, o)
    dmin, o = fields(8, o)
    dmax, o = fields(8, o)
    pref, o = fields(80, o)
    nsamp, o = fields(8, o)

    header_bytes = int(h[184:192])
    n_records = int(h[236:244])
    record_s = float(h[244:252])
    per_rec = [int(x) for x in nsamp]
    if n_records < 0:                             # "-1" = unknown; infer it
        data_bytes = path.stat().st_size - header_bytes
        n_records = data_bytes // (2 * sum(per_rec))

    chans = []
    for i in range(ns):
        p0, p1, d0, d1 = (float(pmin[i]), float(pmax[i]),
                          float(dmin[i]), float(dmax[i]))
        gain = (p1 - p0) / (d1 - d0) if d1 != d0 else 1.0
        chans.append(Channel(
            index=i, label=labels[i], dim=dims[i],
            fs=per_rec[i] / record_s if record_s > 0 else float("nan"),
            n_per_record=per_rec[i], gain=gain, offset=p0 - gain * d0,
            prefilter=pref[i]))
    return EdfHeader(path, n_records, record_s, header_bytes,
                     h[168:176].decode("ascii", "ignore").strip(),
                     h[176:184].decode("ascii", "ignore").strip(), chans)


def read_channel(hdr: EdfHeader, label, microvolts=True):
    """One channel as float64 at its native rate. Returns (signal, fs)."""
    ch = hdr.find(label)
    per_rec = [c.n_per_record for c in hdr.channels]
    rec_len = sum(per_rec)
    start = sum(per_rec[:ch.index])
    raw = np.memmap(hdr.path, dtype="<i2", mode="r", offset=hdr.header_bytes,
                    shape=(hdr.n_records, rec_len))
    x = np.asarray(raw[:, start:start + ch.n_per_record], dtype=np.float64).ravel()
    del raw
    x = x * ch.gain + ch.offset
    if microvolts:
        k = ch.uv_scale
        if k is None:
            raise ValueError(f"{label}: physical unit '{ch.dim}' is not a voltage")
        x *= k
    return x, ch.fs

In [ ]:
%%writefile /content/rbd-rwa/src/fetch.py
"""Resumable downloads. PhysioNet's TLS drops connections mid-transfer, so
every request retries and partial files resume with an HTTP Range header.

Open PhysioNet databases are mirrored byte-for-byte on AWS Open Data
(s3://physionet-open), which is far faster from cloud notebooks than
physionet.org, so the mirror is tried first and PhysioNet is the fallback."""
import time
import urllib.request
from pathlib import Path

CAP_MIRRORS = (
    "https://physionet-open.s3.amazonaws.com/capslpdb/1.0.0/",
    "https://physionet.org/files/capslpdb/1.0.0/",
)
CAP_BASE = CAP_MIRRORS[-1]


def remote_size(url, tries=5):
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method="HEAD")
            return int(urllib.request.urlopen(req, timeout=45)
                       .headers.get("Content-Length", 0))
        except Exception:
            if i == tries - 1:
                return None
            time.sleep(3)


def download(url, dest: Path, tries=10, quiet=False):
    """Fetch `url` to `dest`, resuming if a partial file is already there."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    total = remote_size(url)
    if total is None:
        raise IOError(f"cannot reach {url}")
    if dest.exists() and dest.stat().st_size == total:
        return dest
    for i in range(tries):
        have = dest.stat().st_size if dest.exists() else 0
        if have == total:
            return dest
        if have > total:                          # stale file from another source
            dest.unlink()
            have = 0
        try:
            headers = {"Range": f"bytes={have}-"} if have else {}
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "ab") as f:
                while chunk := r.read(1 << 20):
                    f.write(chunk)
        except Exception as e:
            if not quiet:
                print(f"    {dest.name}: {type(e).__name__}, resuming "
                      f"({dest.stat().st_size / 1e6 if dest.exists() else 0:.0f} MB)")
            time.sleep(3)
    got = dest.stat().st_size if dest.exists() else 0
    if got != total:
        raise IOError(f"{dest.name}: got {got} of {total} bytes")
    return dest


def download_any(name, dest: Path, mirrors=CAP_MIRRORS):
    """Try each mirror in turn; the first one that completes wins."""
    last = None
    for base in mirrors:
        try:
            return download(base + name, dest, quiet=True)
        except Exception as e:
            last = e
    raise IOError(f"{name}: all mirrors failed ({last})")


def cap_records():
    """All CAP record names, grouped by cohort prefix."""
    txt = None
    for base in CAP_MIRRORS:
        try:
            txt = urllib.request.urlopen(base + "RECORDS", timeout=45).read().decode()
            break
        except Exception:
            continue
    if txt is None:
        raise IOError("cannot fetch the CAP RECORDS list from any mirror")
    recs = [r.replace(".edf", "") for r in txt.split()]
    groups = {}
    for r in recs:
        key = "".join(c for c in r if not c.isdigit())
        groups.setdefault(key, []).append(r)
    return groups


def fetch_cap(record, scratch: Path):
    """Download one CAP recording (.edf + .txt) into scratch."""
    edf = download_any(record + ".edf", Path(scratch) / f"{record}.edf")
    ann = download_any(record + ".txt", Path(scratch) / f"{record}.txt")
    return edf, ann

In [ ]:
%%writefile /content/rbd-rwa/src/cap.py
"""Parse CAP Sleep Database annotations and align them to the EDF clock.

The annotation file is a RemLogic export whose epochs are stamped with WALL
CLOCK times ("23:07:19"), while the signal array starts at whatever time the
EDF header declares. Aligning them is the step most likely to go silently
wrong: an unnoticed offset shifts every sleep stage relative to the signal and
quietly corrupts every downstream measurement, so the offset is computed
explicitly here and reported.
"""
import datetime as dt
import re
from pathlib import Path

import numpy as np

# W / S1 / S2 / S3 / S4 / R / MT  ->  compact codes
STAGE_CODE = {"W": 0, "S1": 1, "S2": 2, "S3": 3, "S4": 3, "R": 4}
STAGE_NAME = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM", -1: "excluded"}
EPOCH_SEC = 30


def edf_start_datetime(edf_path: Path) -> dt.datetime:
    """Recording start from the EDF header (bytes 168-183)."""
    with open(edf_path, "rb") as f:
        head = f.read(184)
    d = head[168:176].decode("ascii", "ignore").strip()   # dd.mm.yy
    t = head[176:184].decode("ascii", "ignore").strip()   # hh.mm.ss
    day, mon, yr = (int(x) for x in re.split(r"[.\-/]", d))
    hh, mm, ss = (int(x) for x in re.split(r"[.\-:]", t))
    # EDF year is two digits: 85-99 -> 19xx, 00-84 -> 20xx (EDF spec clipping rule)
    yr += 1900 if yr >= 85 else 2000
    return dt.datetime(yr, mon, day, hh, mm, ss)


def parse_annotations(txt_path: Path):
    """Return (times, stages) for every scored 30 s epoch, in file order."""
    raw = Path(txt_path).read_bytes().decode("latin-1")
    lines = [l.rstrip("\r") for l in raw.split("\n")]

    # The table begins at the header row; column order varies between files.
    hdr_i = next(i for i, l in enumerate(lines) if l.startswith("Sleep Stage"))
    cols = [c.strip() for c in lines[hdr_i].split("\t")]
    i_stage = cols.index("Sleep Stage")
    i_time = next(i for i, c in enumerate(cols) if c.startswith("Time"))
    i_event = cols.index("Event") if "Event" in cols else None

    times, stages = [], []
    for line in lines[hdr_i + 1:]:
        parts = line.split("\t")
        if len(parts) <= max(i_stage, i_time):
            continue
        # Only sleep-stage rows; CAP files also carry MCAP-A1/A2/A3 arousal events
        if i_event is not None and i_event < len(parts):
            if not parts[i_event].strip().upper().startswith("SLEEP"):
                continue
        stage = parts[i_stage].strip().upper()
        tm = parts[i_time].strip()
        m = re.match(r"(\d{1,2})[:.](\d{2})[:.](\d{2})", tm)
        if not m:
            continue
        times.append(tuple(int(g) for g in m.groups()))
        stages.append(STAGE_CODE.get(stage, -1))
    return times, np.array(stages, dtype=np.int8)


def build_hypnogram(edf_path: Path, txt_path: Path, n_seconds: float):
    """
    Hypnogram on the EDF's own timeline: one code per 30 s epoch from recording
    start. Returns (hypnogram, info).
    """
    start = edf_start_datetime(edf_path)
    times, stages = parse_annotations(txt_path)
    if len(times) == 0:
        raise ValueError(f"{Path(txt_path).name}: no scored epochs found")

    # Walk the clock forward, adding a day whenever it wraps past midnight.
    day = 0
    prev = None
    offsets = []
    for hh, mm, ss in times:
        cur = hh * 3600 + mm * 60 + ss
        if prev is not None and cur < prev - 3600:   # wrapped midnight
            day += 1
        prev = cur
        stamp = dt.datetime(start.year, start.month, start.day, hh, mm, ss) \
            + dt.timedelta(days=day)
        offsets.append((stamp - start).total_seconds())

    offsets = np.array(offsets)
    # The first annotation may precede the EDF start by a day if the recording
    # itself began before midnight and the header date is the following day.
    if offsets[0] < -12 * 3600:
        offsets += 24 * 3600

    n_epochs = int(np.floor(n_seconds / EPOCH_SEC))
    hyp = np.full(n_epochs, -1, dtype=np.int8)
    idx = np.round(offsets / EPOCH_SEC).astype(int)
    keep = (idx >= 0) & (idx < n_epochs)
    hyp[idx[keep]] = stages[keep]

    info = {
        "edf_start": start.isoformat(),
        "first_annotation_offset_s": float(offsets[0]),
        "n_annotated": int(len(stages)),
        "n_placed": int(keep.sum()),
        "n_dropped_outside_signal": int((~keep).sum()),
        "n_epochs": int(n_epochs),
        "counts": {STAGE_NAME[k]: int((hyp == k).sum())
                   for k in sorted(STAGE_NAME) if (hyp == k).sum()},
    }
    return hyp, info

In [ ]:
%%writefile /content/rbd-rwa/src/extract.py
"""Turn a raw overnight PSG into a small derived file.

One CAP recording is ~370 MB; what RWA quantification needs is the EMG
amplitude envelope and the hypnogram, which is a few MB. Extracting and
discarding the raw signal is what makes the project runnable on hosted
notebooks, where disk is ephemeral and sessions end without warning.

Per recording we keep, for chin EMG first and then each leg EMG channel:
    rms   (n_channels, n_bins) float16 - RMS amplitude per 20 ms bin
    mav   (n_channels, n_bins) float16 - mean absolute (rectified) amplitude
    hypnogram (n_epochs,) int8         - 30 s sleep stages on the EDF timeline
    meta  json - channels, native rates, filter bands, units, alignment

TIMING. Bins are exactly 1/ENV_FS seconds long whatever the source rate, so a
bin index converts to time without drift. (An earlier version used a fixed
integer number of samples per bin; at 512 Hz that made 51.2 bins per second
instead of 50 and shifted the hypnogram by ~11 minutes by the end of a night.)
"""
import json
from pathlib import Path

import numpy as np

from .config import ENV_FS, EMG_BAND, DERIVED
from .cap import build_hypnogram, EPOCH_SEC
from .edf import read_header, read_channel

# Channel roles. CAP uses Italian labels: DX = destra (right), SX = sinistra (left).
CHIN_PATTERNS = ("EMG1-EMG2", "EMG1", "CHIN", "MENT")
LEG_PATTERNS = ("DX1-DX2", "SX1-SX2", "TIB", "LEG")


def _match(channels, patterns):
    out = []
    for c in channels:
        u = c.label.upper().replace(" ", "")
        if c.uv_scale is not None and any(p.replace(" ", "") in u for p in patterns):
            out.append(c)
    return out


def emg_band(fs, band=EMG_BAND):
    """The EMG pass-band actually usable at this sampling rate."""
    lo, hi = band
    hi = min(hi, fs / 2.0 - 5.0)
    if hi <= lo:
        raise ValueError(f"sampling rate {fs} Hz too low for EMG band {band}")
    return lo, hi


def envelope(sig, fs, env_fs=ENV_FS, band=EMG_BAND, mains=None):
    """
    Band-pass (and optionally notch) the EMG, then reduce it to amplitude per
    bin of exactly 1/env_fs seconds. Returns (rms, mav, (lo, hi)).
    """
    from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt

    lo, hi = emg_band(fs, band)
    sos = butter(4, [lo, hi], btype="band", fs=fs, output="sos")
    x = sosfiltfilt(sos, np.asarray(sig, dtype=np.float64))
    if mains:
        # Mains and its harmonics that fall inside the pass-band
        for f0 in np.arange(mains, hi, mains):
            b, a = iirnotch(f0, Q=30.0, fs=fs)
            x = filtfilt(b, a, x)

    n_bins = int(np.floor(len(x) * env_fs / fs))
    if n_bins == 0:
        raise ValueError("signal shorter than one envelope bin")
    edges = np.floor(np.arange(n_bins + 1) * fs / env_fs).astype(np.int64)
    counts = np.diff(edges)
    rms = np.sqrt(np.add.reduceat(x * x, edges[:-1]) / counts)
    mav = np.add.reduceat(np.abs(x), edges[:-1]) / counts
    return rms.astype(np.float32), mav.astype(np.float32), (lo, hi)


def extract_recording(edf_path: Path, txt_path: Path, record: str,
                      out_dir: Path = None, overwrite=False, mains=50.0):
    """Extract one CAP recording to a compact .npz. Returns the output path."""
    out_dir = Path(out_dir or DERIVED)
    out = out_dir / f"{record}.npz"
    if out.exists() and not overwrite:
        return out

    hdr = read_header(edf_path)
    chin = _match(hdr.channels, CHIN_PATTERNS)
    legs = _match(hdr.channels, LEG_PATTERNS)
    if not chin:
        raise ValueError(f"{record}: no chin EMG channel among "
                         f"{[c.label for c in hdr.channels]}")
    picks = chin[:1] + legs                      # chin first, by convention

    rms, mav, chan_meta = [], [], []
    for c in picks:
        x, fs = read_channel(hdr, c.label)
        r, m, (lo, hi) = envelope(x, fs, mains=mains)
        rms.append(r)
        mav.append(m)
        peak = float(np.max(np.abs(x))) if len(x) else float("nan")
        chan_meta.append({
            "label": c.label, "fs": fs, "band": [lo, hi], "unit": c.dim,
            "prefilter": c.prefilter,
            # share of samples sitting at the recorder's clipping level
            "clipped_frac": float(np.mean(np.abs(x) >= 0.999 * peak)) if peak > 0 else 0.0,
        })
        del x
    n = min(len(r) for r in rms)                 # channels can differ by a bin
    rms = np.stack([r[:n] for r in rms])
    mav = np.stack([m[:n] for m in mav])

    n_seconds = n / ENV_FS
    hyp, align = build_hypnogram(edf_path, txt_path, n_seconds)

    meta = {
        "record": record,
        "channels": [c.label for c in picks],
        "chin_channel": chin[0].label,
        "leg_channels": [c.label for c in legs],
        "channel_info": chan_meta,
        "env_fs": ENV_FS,
        "mains_notch": mains,
        "duration_s": float(n_seconds),
        "epoch_sec": EPOCH_SEC,
        "alignment": align,
    }
    out_dir.mkdir(parents=True, exist_ok=True)
    tmp = out.with_name(out.stem + ".tmp.npz")
    np.savez_compressed(tmp, rms=rms.astype(np.float16), mav=mav.astype(np.float16),
                        hypnogram=hyp, meta=json.dumps(meta))
    tmp.replace(out)                             # never leave a half-written file
    return out


def load_derived(path, kind="rms"):
    """Read a derived file back as (env, hypnogram, meta); env is float32."""
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z["meta"]))
    key = kind if kind in z.files else "env"     # files from the first version
    return z[key].astype(np.float32), z["hypnogram"], meta


def summarise(path):
    env, hyp, meta = load_derived(path)
    rem = int((hyp == 4).sum())
    rates = "/".join(f"{c['fs']:g}" for c in meta.get("channel_info", []))
    return (f"{meta['record']:<8} {meta['duration_s']/3600:>5.1f} h  "
            f"fs {rates} Hz  chin={meta['chin_channel']:<10} "
            f"legs={len(meta['leg_channels'])}  "
            f"REM {rem} epochs ({rem*0.5:.0f} min)")

In [ ]:
%%writefile /content/rbd-rwa/src/rwa.py
"""The published methods for quantifying REM sleep without atonia.

Each method reduces one night to a percentage (or an index) which is compared
against a diagnostic cut-off. They differ along four axes:

    mini-epoch length   1 s (RAI), 2 s (Montreal phasic), 3 s (SINBAR, AASM
                        phasic), 20 s (Montreal tonic), 30 s (AASM tonic)
    muscles             submentalis alone, or submentalis + bilateral flexor
                        digitorum superficialis (SINBAR)
    baseline            "minimum amplitude" during REM or NREM, variously
                        defined; here parameterised explicitly
    threshold           2x or 4x that baseline

Five COMPUTERISED indices (RAI among them) were compared on the same 146
recordings by Cesari et al., Sleep 2018;41(10):zsy133, which reported
discrimination between groups. This module puts the visual-criteria methods
and the RAI on one code path so that all of them can be applied unchanged to
large clinical cohorts.

PARAMETER PROVENANCE. The structure of each method follows its original
description, but several numeric details - notably the exact baseline
definition and the RAI binning - are stated ambiguously or inconsistently
across the primary papers and secondary reviews. Every such value is a named
field on Method so it can be checked against the source and changed in one
place. Anything listed in a method's `verify` tuple has NOT been confirmed
against the original publication and must be before any result is reported.
"""
from dataclasses import dataclass, field

import numpy as np

REM_CODE = 4
NREM_CODES = (2, 3)


# ══════════════════════════════════════════════════════════════════════════════
#  SHARED MACHINERY
# ══════════════════════════════════════════════════════════════════════════════

def mini_epoch_rms(env, env_fs, seconds):
    """Split an envelope into consecutive mini-epochs and take the RMS of each."""
    w = int(round(env_fs * seconds))
    n = len(env) // w
    if n == 0:
        return np.empty(0), w
    return np.sqrt((env[: n * w].reshape(n, w) ** 2).mean(axis=1)), w


def stage_mask_at(env_fs, hypnogram, epoch_sec, codes, n_samples):
    """Boolean mask over envelope samples for the given sleep-stage codes."""
    per = int(round(env_fs * epoch_sec))
    mask = np.zeros(n_samples, dtype=bool)
    for i, code in enumerate(hypnogram):
        if code in codes:
            a = i * per
            if a >= n_samples:
                break
            mask[a: min(a + per, n_samples)] = True
    return mask


def baseline_amplitude(env, env_fs, hypnogram, epoch_sec, how, seconds=1.0):
    """
    Background EMG level against which activity is judged.

    how = "min_nrem" : lowest mini-epoch amplitude in NREM. This is the classic
                       wording, but a literal minimum is set by a single
                       quietest window and is therefore unstable.
          "p5_nrem"  : 5th percentile in NREM - a robust reading of the same
                       intent, and this project's default.
          "min_rem"  : lowest mini-epoch amplitude within REM itself.
          "p5_rem"   : 5th percentile within REM.

    Which of these is used materially changes every downstream percentage, so
    it is treated as an explicit choice rather than an implementation detail.
    """
    codes = NREM_CODES if how.endswith("nrem") else (REM_CODE,)
    m = stage_mask_at(env_fs, hypnogram, epoch_sec, codes, len(env))
    if m.sum() < env_fs * seconds * 10:
        return np.nan
    r, _ = mini_epoch_rms(env[m], env_fs, seconds)
    if len(r) == 0:
        return np.nan
    return float(r.min() if how.startswith("min") else np.percentile(r, 5))


def phasic_bursts(env, env_fs, threshold, min_s=0.1, max_s=5.0):
    """
    Boolean array marking samples that fall inside a phasic burst.

    A burst is a run of consecutive samples above `threshold` whose duration
    lies in [min_s, max_s]. The upper bound is what distinguishes phasic bursts
    from sustained tonic activity, so longer runs are excluded here and counted
    by the tonic measure instead.
    """
    above = env > threshold
    out = np.zeros_like(above)
    if not above.any():
        return out
    edges = np.diff(np.r_[0, above.astype(np.int8), 0])
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    dur = (stops - starts) / env_fs
    keep = (dur >= min_s) & (dur <= max_s)
    for a, b in zip(starts[keep], stops[keep]):
        out[a:b] = True
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  METHOD DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Method:
    name: str
    mini_s: float                 # mini-epoch length for the phasic measure
    tonic_s: float                # epoch length for the tonic measure
    phasic_mult: float            # burst threshold, multiples of baseline
    tonic_mult: float             # tonic threshold, multiples of baseline
    tonic_fraction: float         # fraction of the epoch that must be active
    baseline: str = "p5_nrem"
    muscles: tuple = ("chin",)
    cutoff: float = float("nan")  # published cut-off, on the "any activity" %
    notes: str = ""
    verify: tuple = field(default_factory=tuple)


METHODS = {
    # Lapierre & Montplaisir 1992. Tonic scored on 20 s epochs, phasic on 2 s
    # mini-epochs containing bursts above 4x background.
    "montreal": Method(
        "Montreal", mini_s=2.0, tonic_s=20.0,
        phasic_mult=4.0, tonic_mult=2.0, tonic_fraction=0.50,
        notes="tonic = activity >2x background sustained over half a 20 s epoch",
        verify=("tonic_fraction", "cutoff"),
    ),
    # Frauscher et al. (SINBAR). 3 s mini-epochs. The published 31.9% cut-off
    # applies to the COMBINED submentalis + bilateral FDS montage, not to chin
    # alone, so scoring chin-only against it is not like-for-like.
    "sinbar": Method(
        "SINBAR", mini_s=3.0, tonic_s=30.0,
        phasic_mult=2.0, tonic_mult=2.0, tonic_fraction=0.50,
        muscles=("chin", "legs"), cutoff=31.9,
        notes="'any' activity = tonic or phasic, chin + bilateral FDS; CAP "
              "provides leg (tibialis) rather than FDS, an imperfect substitute",
        verify=("phasic_mult", "muscles"),
    ),
    # AASM scoring manual. Tonic on the full 30 s epoch; a phasic epoch is
    # positive when at least 5 of its ten 3 s mini-epochs contain bursts.
    "aasm": Method(
        "AASM", mini_s=3.0, tonic_s=30.0,
        phasic_mult=4.0, tonic_mult=1.0, tonic_fraction=0.50,
        notes="phasic epoch positive when >=5 of 10 mini-epochs contain bursts",
        verify=("tonic_mult", "cutoff"),
    ),
}


def score_night(env_chin, env_legs, env_fs, hypnogram, epoch_sec, method: Method):
    """Apply one method to one night. Returns a dict of its measures."""
    base = baseline_amplitude(env_chin, env_fs, hypnogram, epoch_sec,
                              method.baseline, seconds=method.mini_s)
    if not np.isfinite(base) or base <= 0:
        return {"method": method.name, "error": "baseline could not be estimated"}

    rem = stage_mask_at(env_fs, hypnogram, epoch_sec, (REM_CODE,), len(env_chin))
    if rem.sum() < env_fs * method.mini_s * 10:
        return {"method": method.name, "error": "insufficient REM"}

    # Phasic: fraction of REM mini-epochs containing a qualifying burst.
    burst = phasic_bursts(env_chin, env_fs, method.phasic_mult * base)
    if "legs" in method.muscles and env_legs is not None and len(env_legs):
        for row in env_legs:
            burst |= phasic_bursts(row, env_fs, method.phasic_mult * base)

    w = int(round(env_fs * method.mini_s))
    idx = np.flatnonzero(rem)
    idx = idx[: (len(idx) // w) * w].reshape(-1, w) if len(idx) >= w \
        else np.empty((0, w), dtype=int)
    tonic_over = (env_chin[idx] > method.tonic_mult * base).mean(axis=1) \
        if len(idx) else np.empty(0)
    phasic_hit = burst[idx].any(axis=1) if len(idx) else np.empty(0, bool)
    phasic_pct = 100.0 * phasic_hit.mean() if len(idx) else float("nan")

    # Tonic: fraction of REM epochs sustained above threshold, on the method's
    # own tonic epoch length rather than the phasic mini-epoch length.
    wt = int(round(env_fs * method.tonic_s))
    idt = np.flatnonzero(rem)
    idt = idt[: (len(idt) // wt) * wt].reshape(-1, wt) if len(idt) >= wt \
        else np.empty((0, wt), dtype=int)
    tonic_pct = (100.0 * ((env_chin[idt] > method.tonic_mult * base).mean(axis=1)
                          >= method.tonic_fraction).mean()) if len(idt) \
        else float("nan")

    # "Any" activity - the quantity the published cut-offs are stated against.
    any_pct = 100.0 * np.mean(phasic_hit | (tonic_over >= method.tonic_fraction)) \
        if len(idx) else float("nan")

    return {
        "method": method.name,
        "baseline_uV": round(base, 3),
        "rem_minutes": round(rem.sum() / env_fs / 60, 1),
        "tonic_pct": round(float(tonic_pct), 2),
        "phasic_pct": round(float(phasic_pct), 2),
        "any_pct": round(float(any_pct), 2),
        "cutoff": method.cutoff,
        "positive": (bool(any_pct >= method.cutoff)
                     if np.isfinite(method.cutoff) else None),
    }


def rem_atonia_index(env_chin, env_fs, hypnogram, epoch_sec, bin_width=1.0,
                     n_bins=20):
    """
    REM Atonia Index (Ferri et al.). Average rectified amplitude per 1 s
    mini-epoch of REM, noise-corrected by subtracting the minimum, then binned
    by amplitude. The index contrasts the proportion of near-silent seconds
    against the proportion showing intermediate activity, running from 0 (no
    atonia) to 1 (complete atonia).

        RAI = A0 / (1 - A1)

    with A0 the fraction of mini-epochs at or below bin_width and A1 the
    fraction in the next bin.

    VERIFY: the normalisation and bin edges differ between the original paper
    and later descriptions. Treat the absolute value as provisional until
    checked against the source; the RANKING across recordings is considerably
    more robust than the value itself.
    """
    rem = stage_mask_at(env_fs, hypnogram, epoch_sec, (REM_CODE,), len(env_chin))
    if rem.sum() < env_fs * 30:
        return {"method": "RAI", "error": "insufficient REM"}
    amp, _ = mini_epoch_rms(env_chin[rem], env_fs, 1.0)
    if len(amp) < 30:
        return {"method": "RAI", "error": "insufficient REM"}
    amp = amp - amp.min()                                   # noise correction
    edges = np.arange(0, (n_bins + 1) * bin_width, bin_width)
    hist, _ = np.histogram(amp, bins=np.r_[edges, np.inf])
    frac = hist / hist.sum()
    a0, a1 = float(frac[0]), float(frac[1])
    rai = a0 / (1.0 - a1) if (1.0 - a1) > 1e-9 else float("nan")
    return {
        "method": "RAI",
        "rai": round(rai, 4),
        "frac_silent": round(a0, 4),
        "rem_minutes": round(rem.sum() / env_fs / 60, 1),
        "cutoff": 0.8,          # RAI < 0.8 is commonly cited as abnormal
        "positive": bool(rai < 0.8) if np.isfinite(rai) else None,
    }


def score_all(env, hypnogram, meta):
    """Run every method on one derived recording."""
    env_fs, epoch_sec = meta["env_fs"], meta["epoch_sec"]
    chin = env[0]
    legs = env[1:] if len(env) > 1 else None
    rows = [score_night(chin, legs, env_fs, hypnogram, epoch_sec, m)
            for m in METHODS.values()]
    rows.append(rem_atonia_index(chin, env_fs, hypnogram, epoch_sec))
    return rows

In [ ]:
%%writefile /content/rbd-rwa/run_extract.py
"""Stream CAP recordings: download -> extract -> discard raw.

Designed for hosted notebooks, where the session can vanish at any moment.
Progress is written to a manifest after every recording, so re-running
continues from where it stopped and never re-downloads finished work. The next
few recordings download in the background while the current one is being
extracted, and raw EDFs are deleted as soon as their derived file exists, so
disk use stays flat no matter how many recordings are processed.

    python run_extract.py                  # every CAP record (default)
    python run_extract.py rbd1 rbd2 n1     # specific records
    python run_extract.py --cohorts rbd n  # whole cohorts by prefix
    python run_extract.py --keep-raw       # keep EDFs (local debugging only)
"""
import argparse
import json
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config
from src.fetch import fetch_cap, cap_records
from src.extract import extract_recording, summarise

MANIFEST = config.MANIFESTS / "cap_extract.json"
PREFETCH = 3          # recordings downloading ahead of the one being extracted


def load_manifest():
    if MANIFEST.exists():
        try:
            return json.loads(MANIFEST.read_text())
        except json.JSONDecodeError:
            pass
    return {"done": {}, "failed": {}}


def save_manifest(m):
    tmp = MANIFEST.with_suffix(".tmp")
    tmp.write_text(json.dumps(m, indent=2))
    tmp.replace(MANIFEST)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("records", nargs="*")
    ap.add_argument("--cohorts", nargs="*", help="cohort prefixes, e.g. rbd n")
    ap.add_argument("--keep-raw", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()

    config.describe()
    groups = cap_records()
    if args.records:
        records = args.records
    elif args.cohorts:
        records = [r for c in args.cohorts for r in groups.get(c, [])]
    else:
        records = [r for g in groups.values() for r in g]
    print(f"\n{len(records)} records queued "
          f"(cohorts available: { {k: len(v) for k, v in groups.items()} })\n")

    m = load_manifest()
    todo = [r for r in records
            if r not in m["done"] or args.overwrite
            or not (config.DERIVED / f"{r}.npz").exists()]
    print(f"{len(m['done'])} already done, {len(todo)} to process\n")

    pool = ThreadPoolExecutor(max_workers=PREFETCH)
    pending = {}

    def prefetch(i):
        if i < len(todo) and i not in pending:
            pending[i] = pool.submit(fetch_cap, todo[i], config.SCRATCH)

    for i in range(PREFETCH):
        prefetch(i)

    t_start = time.time()
    for i, rec in enumerate(todo):
        prefetch(i + PREFETCH)
        t0 = time.time()
        print(f"[{i + 1}/{len(todo)}] {rec}", flush=True)
        edf = txt = None
        try:
            edf, txt = pending.pop(i).result()
            t_dl = time.time() - t0
            out = extract_recording(edf, txt, rec, overwrite=args.overwrite)
            print(f"    {summarise(out)}  (waited {t_dl:.0f}s for download, "
                  f"{time.time() - t0 - t_dl:.0f}s to extract)", flush=True)
            m["done"][rec] = {"file": out.name, "seconds": round(time.time() - t0, 1)}
            m["failed"].pop(rec, None)
        except Exception as e:
            print(f"    FAILED: {type(e).__name__}: {e}", flush=True)
            traceback.print_exc(limit=2)
            m["failed"][rec] = f"{type(e).__name__}: {e}"
        finally:
            if not args.keep_raw:
                # Raw is disposable once the derived file exists (or failed).
                for p in (edf, txt):
                    if p is not None:
                        Path(p).unlink(missing_ok=True)
        save_manifest(m)
    pool.shutdown(wait=True)

    derived = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    size = sum(p.stat().st_size for p in derived)
    print(f"\n{'=' * 70}")
    print(f"  {len(m['done'])} recordings extracted, {len(m['failed'])} failed "
          f"in {(time.time() - t_start) / 60:.1f} min")
    print(f"  derived data: {len(derived)} files, {size / 1e6:.0f} MB total")
    print(f"  location: {config.DERIVED}")
    if m["failed"]:
        print("\n  failures:")
        for k, v in m["failed"].items():
            print(f"    {k}: {v}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_score.py
"""Score every derived recording with every RWA method.

Writes one tidy row per (recording, method) to results/cap_rwa_scores.csv and
prints, per method, the median score in each cohort and how well the score
separates RBD from healthy sleepers (AUC, 0.5 = chance).

    python run_score.py
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.extract import load_derived
from src.rwa import score_all

COHORT_NAMES = {"rbd": "RBD", "n": "healthy", "nfle": "NFLE", "plm": "PLM",
                "ins": "insomnia", "narco": "narcolepsy", "sdb": "SDB",
                "brux": "bruxism"}


def cohort_of(record):
    return COHORT_NAMES.get("".join(c for c in record if not c.isdigit()), "other")


def auc(pos, neg):
    """Probability a random positive outscores a random negative (ties = 1/2)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    diff = pos[:, None] - neg[None, :]
    return float(((diff > 0) + 0.5 * (diff == 0)).mean())


def main():
    files = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    print(f"{len(files)} derived recordings in {config.DERIVED}\n")
    rows = []
    for p in files:
        env, hyp, meta = load_derived(p)
        for r in score_all(env, hyp, meta):
            r = dict(r)
            r["record"] = meta["record"]
            r["cohort"] = cohort_of(meta["record"])
            rows.append(r)
    df = pd.DataFrame(rows)
    out = config.RESULTS / "cap_rwa_scores.csv"
    df.to_csv(out, index=False)
    print(f"saved {len(df)} rows -> {out}\n")

    # One continuous score per method, oriented so that higher = more RWA
    score = np.where(df["method"] == "RAI", 1 - df.get("rai", np.nan),
                     df.get("any_pct", np.nan))
    df = df.assign(score=score)
    order = [c for c in COHORT_NAMES.values() if c in set(df["cohort"])]
    table = df.pivot_table(index="method", columns="cohort", values="score",
                           aggfunc="median")[order]
    print("Median score per cohort (any-activity %, or 1 - RAI):")
    print(table.round(2).to_string())

    print("\nRBD vs healthy separation:")
    for m, g in df.groupby("method"):
        a = auc(g.loc[g.cohort == "RBD", "score"], g.loc[g.cohort == "healthy", "score"])
        n_r, n_h = (g.cohort == "RBD").sum(), (g.cohort == "healthy").sum()
        print(f"  {m:<9} AUC = {a:.3f}   ({n_r} RBD, {n_h} healthy)")


if __name__ == "__main__":
    main()

In [ ]:
import os
assert os.path.isdir('/content/drive/MyDrive'), 'Google Drive is not mounted - run the first cell'
!cd /content/rbd-rwa && python -c "from src import config; config.describe()"

## 1. Extract every CAP recording (about 40 GB streamed, a few hundred MB kept)

In [ ]:
!cd /content/rbd-rwa && python run_extract.py

## 2. Score every recording with every RWA method

In [ ]:
!cd /content/rbd-rwa && python run_score.py